# 04. PDF 로딩 및 미리보기

## 학습 목표

- PDF 파일에서 텍스트를 추출한다.
- 페이지 단위로 `Document`를 생성한다.
- 추출된 텍스트 품질을 확인한다.

## 공통 전제

- 실습 데이터: `../data/공직자_민원응대_핵심_매뉴얼.pdf`
- 기본 모델: `gpt-4o-mini`
- 기본 임베딩 모델: `text-embedding-3-small`
- 벡터DB: Qdrant 로컬 인메모리 모드

> 이 노트북은 `src` 파일을 import하지 않고, 노트북 안의 코드만으로 실행되도록 구성한다.

In [1]:
from pathlib import Path
import re
from pypdf import PdfReader
from langchain_core.documents import Document

PDF_PATH = Path("../data/공직자_민원응대_핵심_매뉴얼.pdf")
print(PDF_PATH.exists(), PDF_PATH)

True ..\data\공직자_민원응대_핵심_매뉴얼.pdf


## 1. 텍스트 정제 함수

PDF에서 추출된 텍스트는 줄바꿈이나 공백이 불규칙할 수 있다.

In [2]:
def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

## 2. 페이지별 텍스트 추출

In [3]:
reader = PdfReader(str(PDF_PATH))

page_docs = []

for page_number, page in enumerate(reader.pages, start=1):
    text = clean_text(page.extract_text() or "")
    if not text:
        continue

    doc = Document(
        page_content=text,
        metadata={
            "source": PDF_PATH.name,
            "page": page_number
        }
    )
    page_docs.append(doc)

print("추출된 페이지 수:", len(page_docs))

추출된 페이지 수: 20


## 3. 추출 결과 확인

In [4]:
for doc in page_docs[:3]:
    print("=" * 80)
    print(doc.metadata)
    print(doc.page_content[:800])

{'source': '공직자_민원응대_핵심_매뉴얼.pdf', 'page': 1}
현장공무원을 위한 
민원응대
핵심매뉴얼
발간등록번호
11-1741000-100065-14
{'source': '공직자_민원응대_핵심_매뉴얼.pdf', 'page': 2}
CONTENTS
민원응대 관련 기본원칙 5
일반적인 민원 응대요령 9
특이민원 응대요령 13
1. 공통사항 15
2. 전화응대 15
 2-1 정당한 사유없는 반복전화 15
 2-2 정당한 사유없는 장시간 통화 16
 2-3 상급자(기관장 등) 통화 요구 16
 2-4 폭언(욕설, 협박, 성희롱 등) 17
3. 대면응대 18
 3-1 폭언(욕설, 협박, 성희롱 등) 18
 3-2 폭행 19
 3-3 집기 또는 물품 등을 파손하는 경우 20
 3-4 위험물을 소지하여 신변을 위협하는 경우 21
Ⅰ
Ⅱ
Ⅲ
4. 온라인 민원 및 문서 민원응대 22 
 4-1 폭언(욕설, 협박, 성희롱 등) 22
 4-2 정당한 사유없는 반복민원 23
민원담당자 회복 및 보호조치 25
질의응답 29
[참고1] 특이민원 의미와 유형 35
[참고2] 위법행위 유형별 적용 법률 36
[참고3] 우울증 또는 스트레스 자가진단법 37
Ⅳ
Ⅴ
※ 본 매뉴얼의 민원 응대요령은 민원공무원이 업무 수행 시 참고할 수 있도록 가이드 라인을 제시한 것 
 입니다. 민원업무 현장에서 다양한 응대 상황이 발생할 수 있으므로 각급 행정기관에서는 본 매뉴얼의 
 내용을 기관별 상황에 맞게 적절히 활용하시기 바랍니다.
{'source': '공직자_민원응대_핵심_매뉴얼.pdf', 'page': 3}
1. 민원응대 기본 방향 7
2. 행정기관의 민원담당자 보호 7
Ⅰ. 민원응대 관련 기본원칙
Ⅰ 민원응대 관련 기본원칙


## 4. 페이지별 본문 길이 확인

In [5]:
import pandas as pd

preview_df = pd.DataFrame([
    {
        "page": doc.metadata["page"],
        "length": len(doc.page_content),
        "preview": doc.page_content[:100].replace("\n", " ")
    }
    for doc in page_docs
])

preview_df

,page,length,preview
0,1,49,현장공무원을 위한 민원응대 핵심매뉴얼 발간등록번호 11-1741000-100065-14
1,2,604,CONTENTS 민원응대 관련 기본원칙 5 일반적인 민원 응대요령 9 특이민원 응대...
2,3,66,1. 민원응대 기본 방향 7 2. 행정기관의 민원담당자 보호 7 Ⅰ. 민원응대 관련...
3,4,846,Ⅰ. 민원응대 관련 기본원칙 | 7 Ⅰ 민원응대 관련 기본원칙 Ⅰ. 민원응대 관련 ...
4,5,63,1. 공통사항 11 2. 방문민원 11 3. 전화민원 12 Ⅱ. 일반적인 민원 응대...
5,6,657,Ⅱ. 일반적인 민원 응대요령 | 11 Ⅱ 일반적인 민원 응대요령 Ⅱ. 일반적인 민원...
6,7,731,12 | 현장공무원을 위한 민원응대 핵심매뉴얼 03 전화민원 • 민원인에게 인사한 ...
7,8,843,Ⅲ. 특이민원 응대요령 | 15 Ⅲ 특이민원 응대요령 Ⅲ. 특이민원 응대요령 01 ...
8,9,1683,Ⅲ. 특이민원 응대요령 | 1716 | 현장공무원을 위한 민원응대 핵심매뉴얼 Ⅲ 특...
9,10,1852,Ⅲ. 특이민원 응대요령 | 1918 | 현장공무원을 위한 민원응대 핵심매뉴얼 Ⅲ 특...


## 핵심 정리

- PDF RAG의 첫 단계는 텍스트 추출 품질 확인이다.
- PDF 문서는 페이지 단위로 먼저 읽은 뒤, 검색에 적합한 청크로 다시 나눈다.
- 페이지 번호는 답변 출처 표시를 위해 metadata로 유지한다.